# atari_200m_deterministic

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load the DQN runs per game and Dopamine's published DQN curves.
AGENTS = {"DQN": "dqn_atari"}
component = EXPERIMENT.component(AGENTS["DQN"])
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP

results = {}
for agent, name in AGENTS.items():
    for game in GAMES:
        runs = load_results(EXPERIMENT, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs
reference = json.loads(Path("dopamine_dqn_reference.json").read_text())

In [ ]:
# Compute each agent's mean return per game, and each reference up to our last frame.
# Dopamine publishes no DQN curve for some games, so those have no reference.
curves = {}
for (agent, game), runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns))

references = {}
for game in {game for _, game in curves} & reference.keys():
    ref_frames = np.asarray(reference[game]["frames"])
    ref_return = np.asarray(reference[game]["mean_return"])
    in_window = ref_frames <= TOTAL_FRAMES
    references[game] = (ref_frames[in_window], ref_return[in_window])

In [ ]:
# Style for the per-game learning-curve plots.
GAME_STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "legend.loc": "upper left",
}
LINES = {
    "DQN": {"color": "tab:blue", "linewidth": 1.75, "label": "DQN"},
}
BAND_ALPHA = 0.2
REFERENCE_LINE = {"color": "black", "linewidth": 2.5, "label": "Dopamine Ref"}
XLABEL = "Frames"
YLABEL = "Return"
YLABEL_KW = {"va": "center", "labelpad": 20}
XTICKS = np.array([50, 100, 150, 200]) * 1_000_000
XTICK_LABELS = [f"{tick // 1_000_000}M" for tick in XTICKS]


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot one learning curve per game: DQN against Dopamine's reference, if any.
def draw(ax, agent, game):
    frames, (mean, low, high) = curves[agent, game]
    line = LINES[agent]
    ax.fill_between(frames, low, high, color=line["color"], alpha=BAND_ALPHA)
    ax.plot(frames, mean, **line)


figures = {}
for game in GAMES:
    agents = [agent for agent in AGENTS if (agent, game) in curves]
    if not agents:
        continue
    with plt.rc_context(GAME_STYLE):
        fig, ax = plt.subplots()
        for agent in agents:
            draw(ax, agent, game)
        if game in references:
            ax.plot(*references[game], **REFERENCE_LINE)
        ax.set(title=title(game), xlabel=XLABEL)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.legend()
        fig.tight_layout()
    figures[game] = fig
plt.show()

In [ ]:
# Style for the all-games grids.
GRID_STYLE = {
    "figure.figsize": (20, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (1, 5)
PANELS = {  # column of each game's panel
    "asterix": 0,
    "atlantis": 1,
    "frogger": 2,
    "tetris": 3,
    "venture": 4,
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.16, 1, 1), "w_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.15, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.07), "ncol": 2}

In [ ]:
# Plot every game in one grid: DQN against Dopamine's reference.
with plt.rc_context(GRID_STYLE):
    fig, axes = plt.subplots(*GRID_SHAPE)
    legend = {}
    for game, column in PANELS.items():
        ax = axes[column]
        ax.set_box_aspect(PANEL_ASPECT)
        if ("DQN", game) not in curves:
            ax.axis("off")
            continue
        draw(ax, "DQN", game)
        if game in references:
            ax.plot(*references[game], **REFERENCE_LINE)
        ax.set_title(title(game))
        ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
        handles, labels = ax.get_legend_handles_labels()
        legend.update(zip(labels, handles, strict=True))
    fig.tight_layout(**GRID_LAYOUT)
    fig.supxlabel(**GRID_XLABEL)
    fig.supylabel(**GRID_YLABEL)
    fig.legend(legend.values(), legend.keys(), **GRID_LEGEND)
grids = {"grid_dopamine_ref": fig}
figures.update(grids)
plt.show()

In [ ]:
# Save every plotted figure as a PDF, and the grids also as PNGs, in the plots dir.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
for name, fig in grids.items():
    fig.savefig(PLOTS_DIR / f"{name}.png", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")